In [0]:
import pyspark.sql.functions as F
from datetime import timedelta

catalog_name = "ecommerce"
fact_table = f"{catalog_name}.gold.gld_fact_order_items"

# Read your existing gold fact table
fact_df = (
    spark.table(fact_table)
    .select(
        "customer_id",
        "transaction_id",
        "transaction_date",
        "channel",
        "coupon_flag",
        "quantity",
        "discount_percent",
        "net_amount",
        "net_amount_inr"
    )
    .dropna(subset=["customer_id", "transaction_id", "transaction_date"])
)

display(fact_df.limit(10))

In [0]:
max_dt = fact_df.agg(F.max("transaction_date").alias("max_dt")).collect()[0]["max_dt"]
cutoff_dt = max_dt - timedelta(days=30)

print("Max transaction date:", max_dt)
print("Cutoff date:", cutoff_dt)

In [0]:
hist_df = fact_df.filter(F.col("transaction_date") <= F.lit(cutoff_dt))
future_df = fact_df.filter(
    (F.col("transaction_date") > F.lit(cutoff_dt)) &
    (F.col("transaction_date") <= F.lit(max_dt))
)

print("Historical rows:", hist_df.count())
print("Future rows:", future_df.count())

In [0]:
customer_features_df = (
    hist_df.groupBy("customer_id")
    .agg(
        F.countDistinct("transaction_id").alias("orders_hist"),
        F.sum("quantity").alias("items_hist"),
        F.sum("net_amount_inr").alias("revenue_hist_inr"),
        F.avg("net_amount_inr").alias("avg_order_value_hist_inr"),
        F.max("transaction_date").alias("last_order_date"),
        F.avg("discount_percent").alias("avg_discount_percent_hist"),
        F.sum("coupon_flag").alias("coupon_orders_hist"),
        F.countDistinct("channel").alias("distinct_channels_hist")
    )
    .withColumn("recency_days", F.datediff(F.lit(cutoff_dt), F.col("last_order_date")))
)

display(customer_features_df.limit(10))

In [0]:
(
    customer_features_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.ml.customer_features")
)

In [0]:
customer_labels_df = (
    future_df.select("customer_id")
    .distinct()
    .withColumn("label_buy_next_30d", F.lit(1))
)

display(customer_labels_df.limit(20))

In [0]:
(
    customer_labels_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.ml.customer_labels")
)

In [0]:
customer_training_base_df = (
    customer_features_df
    .join(customer_labels_df, on="customer_id", how="left")
    .fillna({"label_buy_next_30d": 0})
)

display(customer_training_base_df.limit(20))

In [0]:
(
    customer_training_base_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.ml.customer_training_base")
)

In [0]:
spark.sql(f"""
SELECT label_buy_next_30d, COUNT(*) AS cnt
FROM {catalog_name}.ml.customer_training_base
GROUP BY label_buy_next_30d
ORDER BY label_buy_next_30d
""").display()